# Self-Correcting RAG — scale benchmark (real models)

Runs the full config curve (naive → reranker_baseline → +planning → full +
ablations) on **HotpotQA** with **real open models** (sentence-transformers +
a local instruction-tuned LLM), with a dev/test split for abstention
calibration, bootstrap CIs, and a paired significance test vs. the reranked
baseline. Use a **GPU runtime** (Runtime → Change runtime type → GPU).

Outputs: `results/results.json`, `results/report.md`, and the
accuracy-vs-cost figure.


## 1. Install


In [ ]:
!git clone https://github.com/blankonline/self-correcting-rag.git
%cd self-correcting-rag
!pip -q install -e ".[models]"
!pip -q install datasets matplotlib


## 2. Run the benchmark

Start small (`--n 100`) to sanity-check, then scale to a few hundred with
`--seeds 3` for publishable numbers. A 1.5B instruct model fits a free GPU;
swap `--model` for something larger if you have the VRAM.


In [ ]:
!python examples/run_benchmark.py \
    --backend real \
    --model Qwen/Qwen2.5-1.5B-Instruct \
    --download \
    --n 300 --unanswerable-frac 0.25 --dev-frac 0.2 --seeds 3 \
    --out-dir results


## 3. Inspect the results


In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open('results/report.md').read()))
display(Image('results/accuracy_cost.png'))


## 4. (Optional) Regenerate the paper with these numbers

`results/results.json` is consumed by the paper's results section — copy it
into the repo and rebuild the PDF locally (see `paper/PAPER.md`).


In [ ]:
import json
r = json.load(open('results/results.json'))
print('backend :', r['meta'])
sig = r['comparison'].get('significance')
print('verdict :', sig['verdict'] if sig else 'n/a')
print('delta F1:', sig['delta_mean_f1'] if sig else 'n/a',
      '| p =', sig['p_value_one_sided'] if sig else 'n/a')
